### Task 1.1: Foundation Setup and Basic Agent Creation

Set up a Customer Support Agent prototype using the Strands Agents framework. This prototype serves as your starting point for exploring the complete journey from agent prototype to production-ready solutions.

After completing this task, your agent will have the following basic architecture:

<div style="text-align:left">
    <img src="images/architecture_lab1_strands.png" width="75%"/>
</div>

*Image description: Simple Agent prototype running locally with local tools*

Install dependencies and import all necessary libraries including AWS SDK, AgentCore components, and Strands framework to prepare the development environment.

In [ ]:
import boto3
import json
import uuid
import time
import requests
from boto3.session import Session

# AgentCore imports
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

# Strands imports
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from mcp.client.streamable_http import streamablehttp_client

# Local tools
from lab_helpers.lab1_strands_agent import (
    get_product_info, get_return_policy, get_technical_support, web_search,
    SYSTEM_PROMPT, MODEL_ID
)
from lab_helpers.utils import get_ssm_parameter, put_ssm_parameter
from scripts.utils import get_cognito_client_secret

# Setup
boto_session = Session()
REGION = boto_session.region_name
CUSTOMER_ID = "customer_001"
SESSION_ID = str(uuid.uuid4())

print("✅ Libraries imported successfully!")

Before creating the agent, examine the local tools that will power our customer support capabilities. Open and review `lab_helpers/lab1_strands_agent.py` to understand:

- Tools are defined locally in this file using `@tool` decorator
- The four tool functions and their purposes:
  - get_product_info(): Get product information
  - get_return_policy(): Get return policy for specific product
  - get_technical_support(): Provide technical support guidance
  - web_search(): Search the web for updated information
- How they use mock data (simulating real databases/APIs)
- The system prompt that defines agent behavior

Create a foundational customer support agent that demonstrates core AI capabilities - from understanding queries to executing actions. This agent combines:

- **Foundation Model**: The "brain" that powers reasoning and decision-making
- **System Prompt**: Behavioral instructions that define the agent's personality and service standards
- **Specialized Tools**: The four local tools (product info, return policy, technical support, web search)

When you call the agent, it follows this process:
1. **Query analysis**: Agent analyzes the customer's question
2. **Tool selection**: Agent determines which tool(s) to use (if any)
3. **Tool execution**: Agent calls the appropriate tool with correct parameters
4. **Response synthesis**: Agent combines tool results with its knowledge to create a helpful response
5. **Quality check**: Agent ensures the response meets the standards in the system prompt

In [ ]:
# Create a basic agent with local tools
model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)
basic_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Basic customer support agent ready!")
print("📋 Available tools: Product Info, Return Policy, Technical Support, Web Search")

Test your basic agent to see how it handles customer queries and uses its tools:

In [ ]:
# Test basic agent functionality
print("💬 Testing basic agent...\n")
response = basic_agent("What's the return policy for laptops?")
print("\n" + "="*50 + "\n")

This initial prototype has several limitations that you will address in subsequent tasks:

- **No persistent memory** - Agent forgets customer history and preferences from previous sessions
- **Local tools only** - No shared or enterprise-grade tool integration  
- **No identity management** - Cannot act on behalf of specific users

### Task 1.2: Enhance Your Agent with Memory

A valued customer contacts your support team about an issue with their recent order. They explain their preferences, share their frustration, and work with your agent to resolve the problem. Three weeks later, they contact support again with a related question. But now they have to repeat everything - their preferences, their history, their context - because your agent only remembers the current conversation session, and not previous sessions. This creates:
- **Frustrated customers** who must repeat their information repeatedly
- **Inefficient support** that cannot build on previous interactions
- **Poor customer satisfaction** due to impersonal, generic responses

Amazon Bedrock AgentCore Memory addresses this limitation by providing a managed service that enables AI agents to maintain context over time, remember important facts, and deliver consistent, personalized experiences. AgentCore Memory operates on two levels:
- **Short-Term Memory**: Immediate conversation context and session-based information (handled automatically by the Strands Agent framework)
- **Long-Term Memory**: Persistent information extracted across multiple conversations, including facts, preferences, and summaries (implemented through AgentCore Memory service with USER_PREFERENCE and SEMANTIC strategies)

Transform your prototype into a customer-aware assistant that is capable of the following examples:
- **"Welcome back, Sarah!"** - Instantly recognizes returning customers
- **"Following up on your laptop issue from last month"** - Connects related conversations seamlessly
- **"Based on your purchase history, here's what I recommend"** - Provides personalized suggestions

After completing this task, your agent will have the following architecture with integrated memory capabilities:

<div style="text-align:left">
    <img src="images/architecture_lab2_memory.png" width="75%"/>
</div>

*Image description: Agent enhanced with AgentCore Memory for persistent customer context and personalization*

**Memory Strategies Configuration**: Create a memory resource combining two intelligent strategies:

| Strategy Type | Purpose | Customer Benefit |
|---------------|---------|------------------|
| USER_PREFERENCE | Customer preferences and behaviors | "I remember you prefer..." |
| SEMANTIC | Factual information and context | "Regarding your previous issue..." |

AgentCore Memory uses namespaces to logically group long-term memory messages using the actorId:
- `support/customer/{actorId}/preferences`: for user preference memory strategy
- `support/customer/{actorId}/semantic`: for semantic memory strategy

In [ ]:
# Initialize memory client for AgentCore Memory service
memory_client = MemoryClient(region_name=REGION)
memory_name = "CustomerSupportMemory"

def create_or_get_memory_resource():
    try:
        # Try to get existing memory resource from SSM parameter
        memory_id = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
        memory_client.gmcp_client.get_memory(memoryId=memory_id)
        return memory_id
    except:
        # Create new memory resource with two strategies
        strategies = [
            {
                # USER_PREFERENCE strategy captures customer preferences and behaviors
                StrategyType.USER_PREFERENCE.value: {
                    "name": "CustomerPreferences",
                    "description": "Captures customer preferences and behavior",
                    "namespaces": ["support/customer/{actorId}/preferences"],
                }
            },
            {
                # SEMANTIC strategy stores factual information from conversations
                StrategyType.SEMANTIC.value: {
                    "name": "CustomerSupportSemantic",
                    "description": "Stores facts from conversations",
                    "namespaces": ["support/customer/{actorId}/semantic"],
                }
            },
        ]
        print("Creating AgentCore Memory resources (2-3 minutes)...")
        # Create memory resource and wait for completion
        response = memory_client.create_memory_and_wait(
            name=memory_name,
            description="Customer support agent memory",
            strategies=strategies,
            event_expiry_days=90,  # Memory events expire after 90 days
        )
        memory_id = response["id"]
        # Store memory ID in SSM for future use
        put_ssm_parameter("/app/customersupport/agentcore/memory_id", memory_id)
        return memory_id

memory_id = create_or_get_memory_resource()
print(f"✅ Memory resource ready: {memory_id}")

Simulate a returning customer named "customer_001" who had previous interactions with your support team. This demonstrates how AgentCore Memory automatically transforms individual conversations into rich, persistent customer insights. Load previous customer interactions and watch AgentCore Memory automatically turn them into long-term customer insights.

In [ ]:
# Seed previous customer interactions
previous_interactions = [
    ("I'm having issues with my MacBook Pro overheating during video editing.", "USER"),
    ("I can help with that thermal issue. Your MacBook Pro order #MB-78432 is still under warranty.", "ASSISTANT"),
    ("What's the return policy on gaming headphones? I need low latency for competitive FPS games", "USER"),
    ("For gaming headphones, you have 30 days to return. Since you're into competitive FPS, I'd recommend checking audio latency specs.", "ASSISTANT"),
    ("I need a laptop under $1200 for programming. Prefer 16GB RAM minimum and good Linux compatibility. I like ThinkPad models.", "USER"),
    ("Perfect! For development work, I'd suggest ThinkPad E series or Dell XPS models with excellent Linux support.", "ASSISTANT"),
]

if memory_id:
    memory_client.create_event(
        memory_id=memory_id,
        actor_id=CUSTOMER_ID,
        session_id="previous_session",
        messages=previous_interactions
    )
    print("✅ Customer history seeded successfully")
    print("⏳ Long-term memory processing will begin automatically...")

Strands Agents provides a powerful hook system that enables components to react to or modify agent behavior through strongly-typed event callbacks. This ensures memory operations happen automatically without manual intervention.

Each time a customer interacts with your agent, it will automatically:
- **Personalize the conversation** based on their previous interactions and preferences
- **Add new interactions to memory** to continuously improve future personalization

What your hooks integration does:
- **Before responding**: Automatically retrieve relevant customer context and preferences
- **After responding**: Automatically save the new interaction to AgentCore Memory

Enable Automatic Customer Context with Memory Hooks.

In [ ]:
class CustomerSupportMemoryHooks(HookProvider):
    def __init__(self, memory_id: str, client: MemoryClient, actor_id: str, session_id: str):
        self.memory_id = memory_id
        self.client = client
        self.actor_id = actor_id
        self.session_id = session_id
        self.namespaces = {
            i["type"]: i["namespaces"][0]
            for i in self.client.get_memory_strategies(self.memory_id)
        }

    def retrieve_customer_context(self, event: MessageAddedEvent):
        # Hook that runs before agent responds to retrieve customer context
        messages = event.agent.messages
        # Only process user messages (not tool results)
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_query = messages[-1]["content"][0]["text"]
            
            try:
                all_context = []
                # Retrieve memories from each strategy namespace, both USER_PREFERENCE and SEMANTIC
                for context_type, namespace in self.namespaces.items():
                    memories = self.client.retrieve_memories(
                        memory_id=self.memory_id,
                        namespace=namespace.format(actorId=self.actor_id),
                        query=user_query,
                        top_k=3,  # Get top 3 relevant memories
                    )
                    # Extract text content from memory objects
                    for memory in memories:
                        if isinstance(memory, dict):
                            content = memory.get("content", {})
                            if isinstance(content, dict):
                                text = content.get("text", "").strip()
                                if text:
                                    all_context.append(f"[{context_type.upper()}] {text}")
                
                # Prepend customer context to user message
                if all_context:
                    context_text = "\n".join(all_context)
                    original_text = messages[-1]["content"][0]["text"]
                    messages[-1]["content"][0]["text"] = f"Customer Context:\n{context_text}\n\n{original_text}"
            except Exception as e:
                print(f"Failed to retrieve customer context: {e}")

    def save_support_interaction(self, event: AfterInvocationEvent):
        # Hook that runs after agent responds to save interaction to memory
        try:
            messages = event.agent.messages
            # Only save if we have both user and assistant messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                customer_query = None
                agent_response = None
                
                # Find the most recent user query and assistant response
                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not agent_response:
                        agent_response = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not customer_query and "toolResult" not in msg["content"][0]:
                        customer_query = msg["content"][0]["text"]
                        break
                
                # Save the interaction to AgentCore Memory
                if customer_query and agent_response:
                    self.client.create_event(
                        memory_id=self.memory_id,
                        actor_id=self.actor_id,
                        session_id=self.session_id,
                        messages=[(customer_query, "USER"), (agent_response, "ASSISTANT")],
                    )
        except Exception as e:
            print(f"Failed to save support interaction: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        # Register both hooks with the agent's hook registry
        registry.add_callback(MessageAddedEvent, self.retrieve_customer_context)
        registry.add_callback(AfterInvocationEvent, self.save_support_interaction)

print("✅ Memory hooks defined - Automatic customer personalization enabled!")
print("🧠 Your agent will now remember customers and personalize every interaction")

Create and test your memory-enhanced agent to see how it retrieves customer context and personalizes responses:

In [ ]:
# Create memory-enhanced agent with hooks
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

memory_agent = Agent(
    model=model,
    tools=[get_product_info, get_return_policy, get_technical_support, web_search],
    hooks=[memory_hooks],
    system_prompt=SYSTEM_PROMPT
)

print("✅ Memory-enhanced agent created!")
print("🧠 Agent will automatically retrieve customer context and save interactions")

In [ ]:
# Wait for memory processing to complete
print("⏳ Waiting 60 seconds for memory processing to complete...")
time.sleep(60)

# Test memory recall
print("🧠 Testing memory-enhanced agent...\n")
response = memory_agent("What are my laptop preferences?")
print("\n" + "="*50 + "\n")

### Task 1.3: Scale with Gateway Integration and AgentCore Identity

With memory in place, focus on powerful tools and scale their impact. Great agents need tools that take full advantage of proprietary and third-party APIs and data, which lets those agents get things done for both internal and external customers. However, building, securing, and scaling agent tools is difficult, becoming a significant blocker for customers moving from agent prototypes to real agent business value in production. AgentCore Gateway serves as a connectivity layer that enables AI agents to discover, authenticate, and invoke real-world tools through a unified **Model Context Protocol (MCP)** endpoint. This is crucial for enterprises managing hundreds of APIs, resources, and tools.

Key Benefits:
- **Fully-managed MCP server** solution without infrastructure management
- **Integration of existing APIs** and Lambda functions
- **Uniform interface** across diverse tools
- **Secure authentication** and authorization
- **Semantic tool discovery** and selection

##### What you'll Build:

**Tool Centralization & Reusability:**
- Migrate web search from local tool to centralized AgentCore Gateway
- Integrate existing enterprise Lambda functions (warranty check)
- Create a shared tool infrastructure that multiple agent types can access

**Enterprise-Grade Security:**
- Implement JWT-based authentication with Cognito integration
- Configure secure inbound authorization for gateway access
- Establish identity-based access control for tool usage

This creates a scalable foundation where tools are centrally managed and reusable across multiple agent types, eliminating code duplication and simplifying maintenance.

AgentCore Identity is also involved in this process. It enables AI agents to securely access AWS resources and help deal with in-bound caller authentication by working with Amazon Cognito. It also enables AI agents to securely access third-party tools and services using outbound auth, although this feature of Agentcore Identity is not used in this lab.

<div style="text-align:left">
    <img src="images/architecture_lab3_identity.png" width="75%"/>
</div>

After completing this task, your agent will have the following architecture with integrated gateway capabilities:

<div style="text-align:left">
    <img src="images/architecture_lab3_gateway.png" width="75%"/>
</div>

*Image description: Agent enhanced with AgentCore Gateway for secure, centralized tool management and enterprise integration*

Create the AgentCore Gateway to expose a Lambda function as an MCP-compatible endpoint. To validate callers authorized to invoke your tools, configure **Inbound Authentication** using OAuth authorization - the standard for MCP servers.

### Understanding Gateway Authentication

AgentCore Gateway uses **OAuth 2.0 with JWT tokens** to secure access to your tools. This prevents unauthorized applications from invoking your Lambda functions.

**Key Concepts:**

1. **Authentication Provider**: Amazon Cognito manages identities and issues tokens
2. **Client Credentials**: Your agent uses a client_id and client_secret (like a username/password for applications)
3. **JWT Tokens**: Short-lived tokens that prove the agent is authorized
4. **Allowed Clients**: An allowlist of client IDs that can access the Gateway

**How It Works:**
```
Agent → Cognito: "Here's my client_id and client_secret"
Cognito → Agent: "Here's your JWT access token"
Agent → Gateway: "Here's my token"
Gateway → Cognito: "Is this token valid and from an allowed client?"
Gateway → Agent: "Access granted"
```

**Security Note**: These credentials have been precreated for you and saved securely in SSM Parameter Store. Credentials should never be hardcoded in your code

In [ ]:
# Retrieve authentication configuration from SSM Parameter Store
# These values were created by the CloudFormation template

# Client ID: Identifies which application is making the request
machine_client_id = get_ssm_parameter("/app/customersupport/agentcore/machine_client_id")
print(f"Machine Client ID: {machine_client_id}")

# Discovery URL: Tells the Gateway where to find Cognito's OAuth configuration
# This URL provides metadata about token endpoints, supported scopes, etc.
cognito_discovery_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
print(f"Discovery URL: {cognito_discovery_url}")

# Configure JWT-based authentication for the Gateway
auth_config = {
    "customJWTAuthorizer": {
        # Only tokens from this client ID will be accepted
        "allowedClients": [machine_client_id],
        # Gateway will fetch OAuth metadata from this URL
        "discoveryUrl": cognito_discovery_url
    }
}

print("✅ Authentication configuration ready")

### Create the AgentCore Gateway

The Gateway acts as a secure proxy between your agent and backend Lambda function. Think of it as an API gateway specifically designed for AI agents.

**What we're creating:**
- **Gateway Infrastructure**: The core Gateway resource
- **MCP Protocol**: Standard protocol for tool communication
- **JWT Authorization**: Using the auth config we just created
- **IAM Role**: Permissions to invoke the Lambda function

**What happens next:**
1. Create the Gateway (this cell)
2. Add Lambda target with tool definitions (next cell)
   - One Lambda function handles multiple tools: `check_warranty_status` and `web_search`
   - The Gateway passes the tool name to Lambda, which routes to the appropriate handler
3. Connect your agent to the Gateway

**Note**: The CloudFormation template deployed a single Lambda function (`CustomerSupportLambda`) that can handle multiple tool operations. This is more efficient than deploying separate Lambda functions for each tool.

In [ ]:
class CreationFailedError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
gateway_name = "customersupport-gw"

try:
    print(f"Creating gateway: {gateway_name}")
    create_response = gateway_client.create_gateway(
        name=gateway_name,
        roleArn=get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role"),
        protocolType="MCP",  # Model Context Protocol
        authorizerType="CUSTOM_JWT",  # Use JWT tokens for auth
        authorizerConfiguration=auth_config,  # Our auth config from above
        description="Customer Support AgentCore Gateway",
    )
    gateway_id = create_response["gatewayId"]
    gateway_url = create_response["gatewayUrl"]
    put_ssm_parameter("/app/customersupport/agentcore/gateway_id", gateway_id)

    # Wait for Gateway to be ready
    print("Waiting for Gateway to be ready...")
    while True:
        status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
        if status == 'READY':
            break
        elif status == 'FAILED':
            raise CreationFailedError("Gateway creation failed")
        else:
            print(f"  Status: {status}")
            time.sleep(5)

    print(f"✅ Gateway created successfully!")
    print(f"   Gateway ID: {gateway_id}")
    print(f"   Gateway URL: {gateway_url}")
    
except gateway_client.exceptions.ConflictException:
    # Gateway already exists, retrieve it
    gateway_id = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = gateway_response["gatewayUrl"]
    print(f"✅ Using existing gateway: {gateway_id}")
    
except CreationFailedError:
    print("\033[31m❌ Gateway creation failed. Check CloudWatch logs for details.\033[0m")

AgentCore Gateway populates the Lambda context with the name of the tool to invoke, while the parameters passed to the tool are provided in the Lambda event. This allows you to integrate existing enterprise Lambda functions (in this case, `AgentCoreLab-CustomerSupportLambda`) that can be reused across multiple agents.

Add your Lambda functions as gateway targets using the API specification:

In [ ]:
# Load API specification for Lambda tools
api_spec = [
    {
        "name": "check_warranty_status",
        "description": "Check warranty status using serial number and email",
        "inputSchema": {
            "type": "object",
            "properties": {
                "serial_number": {"type": "string"},
                "customer_email": {"type": "string"}
            },
            "required": ["serial_number"]
        }
    },
    {
        "name": "web_search",
        "description": "Search the web for updated information",
        "inputSchema": {
            "type": "object",
            "properties": {
                "keywords": {"type": "string", "description": "Search query keywords"},
                "region": {"type": "string", "description": "Search region (e.g., us-en)"},
                "max_results": {"type": "integer", "description": "Maximum results"}
            },
            "required": ["keywords"]
        }
    }
]

# Create gateway target
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": get_ssm_parameter("/app/customersupport/agentcore/lambda_arn"),
            "toolSchema": {"inlinePayload": api_spec},
        }
    }
}

try:
    create_target_response = gateway_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="LambdaTarget",
        description="Lambda tools for customer support",
        targetConfiguration=lambda_target_config,
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    print(f"✅ Gateway target created: {create_target_response['targetId']}")
except Exception as e:
    print(f"Gateway target may already exist: {str(e)}")

Integrate the authentication token from Cognito into an MCPClient from Strands SDK to create a secure MCP connection.

Create an authenticated MCP client to access gateway tools:

In [ ]:
def get_cognito_client_secret():
    # Get Cognito client secret using Cognito API
    client = boto3.client("cognito-idp")
    response = client.describe_user_pool_client(
        UserPoolId=get_ssm_parameter("/app/customersupport/agentcore/userpool_id"),
        ClientId=get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
    )
    return response["UserPoolClient"]["ClientSecret"]

def get_oauth_token():
    # Get OAuth token for gateway authentication using client credentials flow
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "grant_type": "client_credentials",  # OAuth 2.0 client credentials flow
        "client_id": get_ssm_parameter("/app/customersupport/agentcore/machine_client_id"),
        "client_secret": get_cognito_client_secret(),
        "scope": get_ssm_parameter("/app/customersupport/agentcore/cognito_auth_scope"),
    }
    # Request access token from Cognito
    response = requests.post(
        get_ssm_parameter("/app/customersupport/agentcore/cognito_token_url"),
        headers=headers, data=data
    )
    return response.json()

# Get OAuth access token (JWT format)
token_response = get_oauth_token()
access_token = token_response['access_token']

# Create MCP client with Bearer token authentication
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        gateway_url,
        headers={"Authorization": f"Bearer {access_token}"},  # JWT token in Authorization header
    )
)

print(f"✅ MCP client configured for gateway: {gateway_url}")

Combine everything - memory hooks + local tools + gateway tools. This creates a hybrid architecture where some tools remain local (for speed and simplicity) while others are centralized through the gateway (for reusability and enterprise integration).

This approach eliminates code duplication across different agents and creates centralized management for tool updates:

In [ ]:
# Initialize memory hooks for customer context
memory_hooks = CustomerSupportMemoryHooks(memory_id, memory_client, CUSTOMER_ID, SESSION_ID)

# Start MCP client connection to gateway
mcp_client.start()
# Retrieve available tools from the gateway
gateway_tools = mcp_client.list_tools_sync()

# Combine local tools with centralized gateway tools
all_tools = [
    get_product_info,      # Local tool
    get_return_policy,     # Local tool
    get_technical_support, # Local tool
] + gateway_tools          # Gateway tools - web_search, check_warranty_status

# Create enhanced agent with memory and gateway integration
enhanced_agent = Agent(
    model=model,
    tools=all_tools,           # Local + gateway tools
    hooks=[memory_hooks],      # Automatic memory operations
    system_prompt=SYSTEM_PROMPT
)

print("✅ Enhanced Customer Support Agent created!")
print(f"📊 Total tools available: {len(all_tools)}")
print(f"🧠 Memory enabled with ID: {memory_id}")
print(f"🔒 Secure gateway integration: {gateway_url}")

Test your agent with memory and gateway capabilities. Verify that the agent can seamlessly use both local tools and centralized gateway tools, while maintaining customer context through memory.

Test scenarios include warranty checking, web search, and combined memory + gateway capabilities:

In [ ]:
# Test gateway tools
print("🔍 Testing gateway web search...\n")
response2 = enhanced_agent("Search for latest iPhone 15 troubleshooting tips")
print("\n" + "="*50 + "\n")

In [ ]:
# Test warranty check
print("🛡️ Testing warranty check...\n")
response3 = enhanced_agent("Check warranty status for serial number ABC12345678")
print("\n" + "="*50 + "\n")

In [ ]:
# Test combined capabilities
print("🎯 Testing combined memory + gateway capabilities...\n")
response4 = enhanced_agent("I need gaming headphones again, and also search for the latest reviews")
print("\n" + "="*50 + "\n")

## Next Steps

🎉 **Congratulations!** You have completed the notebook exercises.

You have successfully:
- Created a basic AI agent prototype using Strands
- Enhanced it with AgentCore Memory for persistent customer context
- Integrated AgentCore Gateway for secure, centralized tool sharing
- Tested the complete production-ready customer support system

### What's Next?

1. **Close this notebook file**
2. **Return to the lab instructions**
3. **Continue with Task 2** to explore the AgentCore dashboard and see your resources in action


In [23]:
# Cleanup: delete AgentCore Gateway and AgentCore Memory resources
from botocore.exceptions import ClientError

gateway_id_to_delete = globals().get("gateway_id")
memory_id_to_delete = globals().get("memory_id")

if not gateway_id_to_delete:
    try:
        gateway_id_to_delete = get_ssm_parameter("/app/customersupport/agentcore/gateway_id")
    except Exception:
        gateway_id_to_delete = None

if not memory_id_to_delete:
    try:
        memory_id_to_delete = get_ssm_parameter("/app/customersupport/agentcore/memory_id")
    except Exception:
        memory_id_to_delete = None

print(f"Gateway ID: {gateway_id_to_delete}")
print(f"Memory ID: {memory_id_to_delete}")

def _gateway_exists(gateway_identifier: str) -> bool:
    try:
        gateway_client.get_gateway(gatewayIdentifier=gateway_identifier)
        return True
    except ClientError as err:
        error_code = err.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "ResourceNotFoundException":
            return False
        raise

def _delete_all_targets(gateway_identifier: str):
    while True:
        page = gateway_client.list_gateway_targets(
            gatewayIdentifier=gateway_identifier,
            maxResults=100,
        )
        items = page.get("items", [])
        if not items:
            return
        for item in items:
            target_id = item["targetId"]
            print(f"   - Deleting target: {target_id}")
            gateway_client.delete_gateway_target(
                gatewayIdentifier=gateway_identifier,
                targetId=target_id,
            )
        time.sleep(3)

# 1) Delete gateway targets, then gateway (retry + verify)
if gateway_id_to_delete:
    if not _gateway_exists(gateway_id_to_delete):
        print("\nℹ️ Gateway not found before deletion attempt.")
    else:
        print(f"\n🗑️ Deleting gateway targets for: {gateway_id_to_delete}")
        _delete_all_targets(gateway_id_to_delete)

        print(f"🗑️ Deleting gateway: {gateway_id_to_delete}")
        deleted = False
        last_error = None

        for attempt in range(1, 11):
            try:
                gateway_client.delete_gateway(gatewayIdentifier=gateway_id_to_delete)
            except ClientError as err:
                error_code = err.response.get("Error", {}).get("Code", "Unknown")
                error_message = err.response.get("Error", {}).get("Message", "")
                last_error = f"{error_code}: {error_message}"

                # Retry on transient/ordering issues, but do NOT claim success
                if error_code in {"ValidationException", "ConflictException", "ThrottlingException"}:
                    print(f"   Attempt {attempt}/10: delete not accepted yet ({error_code}), retrying...")
                    _delete_all_targets(gateway_id_to_delete)
                    time.sleep(5)
                    continue
                raise

            # Delete request accepted; now verify resource is gone
            for _ in range(18):  # up to ~90 seconds
                if not _gateway_exists(gateway_id_to_delete):
                    deleted = True
                    break
                time.sleep(5)
            if deleted:
                break
            print(f"   Attempt {attempt}/10: still exists after delete request, retrying...")
            _delete_all_targets(gateway_id_to_delete)

        if deleted:
            print("✅ Gateway deleted")
        else:
            raise RuntimeError(
                "Gateway deletion did not complete after retries."
                + (f" Last error: {last_error}" if last_error else "")
            )
else:
    print("\nℹ️ No gateway ID found. Skipping gateway deletion.")

# 2) Delete memory
if memory_id_to_delete:
    try:
        print(f"\n🗑️ Deleting memory: {memory_id_to_delete}")
        memory_client.delete_memory(memory_id=memory_id_to_delete)
        print("✅ Memory deleted")
    except ClientError as err:
        error_code = err.response.get("Error", {}).get("Code", "Unknown")
        if error_code == "ResourceNotFoundException":
            print("ℹ️ Memory not found or already deleted")
        else:
            raise
else:
    print("\nℹ️ No memory ID found. Skipping memory deletion.")

print("\n🎉 Cleanup complete.")

Gateway ID: customersupport-gw-pgyu2qrwwh
Memory ID: CustomerSupportMemory-I25dYS3OxQ

🗑️ Deleting gateway targets for: customersupport-gw-pgyu2qrwwh
🗑️ Deleting gateway: customersupport-gw-pgyu2qrwwh
✅ Gateway deleted

🗑️ Deleting memory: CustomerSupportMemory-I25dYS3OxQ


Failed to delete memory: An error occurred (ResourceNotFoundException) when calling the DeleteMemory operation: Resource not found during DeleteMemory: Failed to retrieve memory with ID: CustomerSupportMemory-I25dYS3OxQ for account: 011673140073


ℹ️ Memory not found or already deleted

🎉 Cleanup complete.
